# GNN+HGB 결합 실험 조건과 입력

저장된 실제 실험 조건을 읽습니다. 이 노트북은 새 학습이나 GPU 작업을 시작하지 않습니다.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, HTML
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "gnn_hgb_fusion"]
ROOT = next((p.resolve() for p in candidates if (p / "PACKAGE.json").exists()
             and json.loads((p / "PACKAGE.json").read_text())["package"] == "gnn_hgb_fusion"), None)
if ROOT is None:
    raise FileNotFoundError("압축을 풀고 해당 패키지의 notebooks 폴더에서 실행하세요.")
sys.path.insert(0, str(ROOT / "src"))
print("패키지:", ROOT)


In [ ]:
display(pd.read_csv(ROOT / 'results/model_comparison.csv'))
for name in ['salt_a3_hgb', 'multi_fraudgt_hgb']:
    fit = json.loads((ROOT / 'results' / name / 'fit.json').read_text())
    print(name, 'fit rows:', fit['rows'], 'positive:', fit['positives'], 'seconds:', fit['seconds'])

## 점수 준비

입력은 `[HGB corrected logit, GNN raw logit]`입니다. HGB 확률은 `[1,100,100]`으로 한 번 보정합니다. 이미 보정된 logit에 다시 보정하지 않습니다. 실제 fit/Val/Test 거래 ID를 점검한 뒤 결합합니다.

In [ ]:
from fusion_workflow import corrected_hgb_logits, load_scores, check_disjoint
# 실제 HGB 모델의 predict_proba 결과가 있을 때:
# hgb_logits = corrected_hgb_logits(hgb_probabilities)
# 외부 점수 준비: tools/export_original_scores.py 및 docs/외부입력_재실행.md 참조

## 해석 주의

Multi의 기반 모델 선택 구간은 결합 fit/Validation과 겹칩니다. 기존 모델 재사용을 평가한 탐색 실험입니다. 결합기 fit 시간은 전체 GNN 추론 시간을 포함하지 않습니다.